# TADA Inference Demo

This notebook demonstrates text-to-speech generation with [TADA](https://huggingface.co/HumeAI/tada-1b), covering:

1. **Basic generation** with default settings
2. **InferenceOptions** — tuning CFG, flow matching, sampling
3. **Negative condition sources** — `negative_step_output`, `prompt`, `zero`
4. **Rejection sampling with speaker verification** — ranking candidates by speaker similarity
5. **Other scorers** — `likelihood`, `duration_median`
6. **Speed-up factor** — two-pass generation with scaled durations
7. **Multilingual** — loading language-specific aligners

## 1. Setup

In [1]:
import torch
import torchaudio
from IPython.display import Audio, display

from tada.modules.encoder import Encoder, EncoderOutput
from tada.modules.tada import TadaForCausalLM, InferenceOptions

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

/mnt/weka/sharath/anaconda3/envs/tada/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


## 2. Load Models

In [2]:
encoder = Encoder.from_pretrained("HumeAI/tada-codec").to(device)
model = TadaForCausalLM.from_pretrained("HumeAI/tada-3b-ml").to(device)
model.decoder.to(device)

Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.27s/it]
Some weights of the model checkpoint at HumeAI/tada-3b-ml were not used when initializing TadaForCausalLM: ['_decoder.decoder_proj.bias', '_decoder.decoder_proj.weight', '_decoder.local_attention_decoder.final_norm.bias', '_decoder.local_attention_decoder.final_norm.weight', '_decoder.local_attention_decoder.layers.0.ffn.0.bias', '_decoder.local_attention_decoder.layers.0.ffn.0.weight', '_decoder.local_attention_decoder.layers.0.ffn.3.bias', '_decoder.local_attention_decoder.layers.0.ffn.3.weight', '_decoder.local_attention_decoder.layers.0.norm.bias', '_decoder.local_attention_decoder.layers.0.norm.weight', '_decoder.local_attention_decoder.layers.0.self_attn._precomputed_mask', '_decoder.local_attention_decoder.layers.0.self_attn.layer_norm.bias', '_decoder.local_attention_decoder.layers.0.self_attn.layer_norm.weight', '_decoder.local_attention_decoder.layers.0.self_attn.out_proj.bias', '_decoder.local_attention

Decoder(
  (decoder_proj): Linear(in_features=512, out_features=1024, bias=True)
  (local_attention_decoder): LocalAttentionEncoder(
    (layers): ModuleList(
      (0-5): 6 x LocalAttentionEncoderLayer(
        (self_attn): LocalSelfAttention(
          (qkv): Linear(in_features=1024, out_features=3072, bias=True)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        )
        (ffn): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.1, inplace=False)
          (3): Linear(in_features=4096, out_features=1024, bias=True)
          (4): Dropout(p=0.1, inplace=False)
        )
        (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      )
    )
    (final_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
 

## 3. Encode Reference Audio

TADA is a voice-cloning TTS model. Provide a reference audio clip to define the speaker voice.

In [3]:
# Load reference audio (replace with your own)
from tada.utils.test_utils import get_sample_dir
import os

audio, sample_rate = torchaudio.load(os.path.join(get_sample_dir(), "ljspeech.wav"))

audio = audio.mean(dim=0, keepdim=True)  # mono
audio = audio / audio.abs().max()        # normalize
audio = audio.to(device)

# Encode: extracts continuous acoustic tokens + text alignment
prompt = encoder(audio, sample_rate=sample_rate)

print(f"Reference audio: {audio.shape[-1] / sample_rate:.1f}s, {prompt.token_values.shape[1]} tokens")
display(Audio(audio[0].cpu(), rate=sample_rate))

🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
Reference audio: 7.6s, 20 tokens


## 4. Basic Text-to-Speech

Generate speech with default `InferenceOptions`. The model auto-regressively predicts text tokens
and continuous acoustic tokens via flow matching.

In [4]:
text = "Hello! This is a demonstration of the TADA text to speech model."

output = model.generate(
    prompt=prompt,
    text=text,
    num_transition_steps=5,
    num_extra_steps=0,
)

display(Audio(output.audio[0].cpu(), rate=24000))

## 5. InferenceOptions

Key parameters that control generation quality:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `acoustic_cfg_scale` | 1.6 | Classifier-free guidance scale for acoustics |
| `duration_cfg_scale` | 1.0 | CFG scale for duration prediction |
| `cfg_schedule` | `"cosine"` | How CFG scale varies over ODE steps: `constant`, `linear`, `cosine` |
| `num_flow_matching_steps` | 20 | Number of Euler ODE steps (more = higher quality, slower) |
| `noise_temperature` | 0.9 | Scale of initial noise for flow matching |
| `time_schedule` | `"logsnr"` | ODE time discretization: `uniform`, `cosine`, `logsnr` |
| `text_temperature` | 0.6 | Sampling temperature for text tokens |
| `text_top_p` | 0.9 | Nucleus sampling threshold |
| `text_repetition_penalty` | 1.1 | Penalizes repeating tokens already in context |

## 6. Negative Condition Sources

Classifier-free guidance requires a "negative" condition. Three sources are available:

- **`"negative_step_output"`** (default): Runs a parallel forward pass with text tokens replaced by padding, using the resulting hidden states as the negative condition. Most expressive but ~2x compute.
- **`"prompt"`**: Reuses hidden states from the prompt prefill as the negative condition. Cheaper than `negative_step_output`, still effective.
- **`"zero"`**: Uses zero vectors as the negative condition. Cheapest, simplest.

In [6]:
# Compare negative condition sources
for neg_source in ["negative_step_output", "prompt", "zero"]:
    opts = InferenceOptions(
        negative_condition_source=neg_source,
    )
    output_neg = model.generate(
        prompt=prompt,
        text=text,
        num_transition_steps=5,
        inference_options=opts,
    )
    print(f"negative_condition_source={neg_source}:")
    display(Audio(output_neg.audio[0].cpu(), rate=24000))

negative_condition_source=negative_step_output:


negative_condition_source=prompt:


negative_condition_source=zero:


## 7. Rejection Sampling with Speaker Verification

At each generation step, generate multiple acoustic candidates and select the best one.

Controlled by:
- `num_acoustic_candidates`: Number of candidates per step (1 = off, default)
- `scorer`: Ranking strategy (`"spkr_verification"`, `"likelihood"`, `"duration_median"`)
- `spkr_verification_weight`: Weight for speaker similarity score (default 1.0)

In [ ]:
# Speaker verification scorer (auto-loads HumeAI/tada-codec (spkr-verf subfolder) on first use)
opts_sv = InferenceOptions(
    acoustic_cfg_scale=1.8,
    duration_cfg_scale=1.0,
    num_flow_matching_steps=20,
    num_acoustic_candidates=8,
    scorer="spkr_verification",
    spkr_verification_weight=1.0,
    time_schedule="logsnr",
)

output_sv = model.generate(
    prompt=prompt,
    text=text,
    num_transition_steps=5,
    inference_options=opts_sv,
)

print("With speaker verification (8 candidates):")
display(Audio(output_sv.audio[0].cpu(), rate=24000))

With speaker verification (4 candidates):


## 8. Other Scoring Strategies

- **`"likelihood"`**: Ranks candidates by conditional reconstruction loss (how well the ODE trajectory
  explains the candidate). No external model needed.
- **`"duration_median"`**: Picks the candidate whose predicted duration is closest to the median
  across all candidates. Good for duration stability.

In [9]:
# Likelihood scorer
opts_ll = InferenceOptions(
    num_acoustic_candidates=4,
    scorer="likelihood",
)

output_ll = model.generate(
    prompt=prompt,
    text=text,
    num_transition_steps=5,
    inference_options=opts_ll,
)

print("Likelihood scorer (4 candidates):")
display(Audio(output_ll.audio[0].cpu(), rate=24000))

Likelihood scorer (4 candidates):


In [10]:
# Duration median scorer
opts_dm = InferenceOptions(
    num_acoustic_candidates=4,
    scorer="duration_median",
)

output_dm = model.generate(
    prompt=prompt,
    text=text,
    num_transition_steps=5,
    inference_options=opts_dm,
)

print("Duration median scorer (4 candidates):")
display(Audio(output_dm.audio[0].cpu(), rate=24000))

Duration median scorer (4 candidates):


## 9. Speed-Up Factor (Two-Pass Generation)

The `speed_up_factor` option runs generation in two passes:
1. **First pass**: Generate freely to get natural duration predictions
2. **Second pass**: Re-generate with durations scaled by `1/speed_up_factor`, using `forced_time`

Values > 1.0 speed up speech, < 1.0 slow it down.

In [10]:
# Speed up by 20%
opts_fast_speech = InferenceOptions(
    speed_up_factor=1.2,
    acoustic_cfg_scale=1.6,
)

output_fast_speech = model.generate(
    prompt=prompt,
    text=text,
    num_transition_steps=5,
    inference_options=opts_fast_speech,
)

print("Speed up factor 1.2 (20% faster speech):")
display(Audio(output_fast_speech.audio[0].cpu(), rate=24000))

Speed up factor 1.2 (20% faster speech):


## 10. Multilingual Support

TADA supports multiple languages via language-specific aligners. Pass `language` to
`Encoder.from_pretrained()` to load the appropriate aligner.

Available languages: `ar`, `ch`, `de`, `es`, `fr`, `it`, `ja`, `pl`, `pt`

(Default `language=None` loads the English aligner.)

In [11]:
# Example: French
encoder_fr = Encoder.from_pretrained("HumeAI/tada-codec", language="fr").to(device)

# Use a French reference audio and text
# audio_fr, sr_fr = torchaudio.load("path/to/french_reference.wav")
# audio_fr = audio_fr.mean(dim=0, keepdim=True).to(device)
# prompt_fr = encoder_fr(audio_fr, sample_rate=sr_fr)
#
# output_fr = model.generate(
#     prompt=prompt_fr,
#     text="Bonjour, ceci est une demonstration du modele TADA.",
#     num_transition_steps=5,
# )
# display(Audio(output_fr.audio[0].cpu(), rate=24000))

print("French encoder loaded successfully:", type(encoder_fr))

French encoder loaded successfully: <class 'tada.modules.encoder.Encoder'>


## 11. Zero-Shot Generation (No Reference Audio)

You can also generate without a reference audio prompt using `EncoderOutput.empty()`.
The model will use a random voice.

In [12]:
output_zs = model.generate(
    prompt=EncoderOutput.empty(device),
    text="This is zero shot generation without any reference audio.",
    num_transition_steps=0,
    num_extra_steps=50,
)

print("Zero-shot (no prompt):")
display(Audio(output_zs.audio[0].cpu(), rate=24000))

Zero-shot (no prompt):
